                # Ensemble

                Compile three different few-shot programs and reduce their boolean predictions with majority voting.

                **Use it when:** Errors are costly enough to justify several model calls per prediction and component diversity improves robustness.

                **What compilation changes:** Combines LabeledFewShot, BootstrapFewShot, and BootstrapRS; it does not learn a single new prompt.

                Important configuration in this benchmark:

                - fixed pool of three component programs
- typed-boolean majority reducer
- all components execute for every test example

                The 74-row dataset and pair-grouped train/validation/test membership are frozen.
                The test partition is deliberately baseline-adversarial, so these scores teach
                optimizer tradeoffs; they are not a general-purpose AI-detector leaderboard.

In [1]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "chapter06").is_dir() else cwd.parent
if not (REPO_ROOT / "chapter06" / "results" / "benchmark_summary.json").exists():
    raise RuntimeError("Run this notebook from the repository or chapter06 directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from chapter06.notebook_support import (
    artifact_paths,
    benchmark_snapshot,
    learned_program_preview,
    verify_prompt_artifact,
)

OPTIMIZER = 'ensemble'
print(f"optimizer={OPTIMIZER!r}")
print("reading the checked-in chapter result; no API calls")

optimizer='ensemble'
reading the checked-in chapter result; no API calls


                ## Compile shape

                This is the essential DSPy call used by the shared runner (setup variables omitted):

                ```python
                programs = [labeled_program, bootstrapped_program, searched_program]
optimizer = dspy.Ensemble(reduce_fn=boolean_majority)
optimized_detector = optimizer.compile(programs)
                ```

                `compile` returns a program. Calling that program on the untouched test examples is
                a separate phase; the notebook reports optimization cost/time separately from inference latency.

In [2]:
print(benchmark_snapshot(OPTIMIZER))
print()
print(artifact_paths(OPTIMIZER))

Ensemble — frozen full-profile rerun
status: completed
task model: openai/gpt-5.6-luna
test accuracy: 80.0%
uplift: +30.0 points vs Luna reference
optimization: $0.3028 and 6.7min
inference latency: mean 5.10s; p95 6.35s
reload checks: prompt=True; model=None; predictions=3/3 labels

Published artifacts:
- canonical program snapshot: chapter06/optimized_programs/final/ensemble.json
- canonical prompt: chapter06/results/final_prompts/ensemble.json
- chapter comparison: chapter06/CHAPTER_RESULTS.md


## Read the result

Accuracy is only half the result: compare mean and p95 inference latency with single-program optimizers because every prediction fans out to three calls.

The next cell shows a bounded readable preview. The complete, lossless prompt and
saved program snapshot remain at the paths printed above.

In [3]:
print(learned_program_preview(OPTIMIZER))
print()
print("Frozen program/prompt consistency:", verify_prompt_artifact(OPTIMIZER))

Predictor: programs[0].detect.predict
Learned instruction (59 characters):
Decide whether the supplied passage was generated by an AI.

Demonstrations: 4
1. is_ai=False: join is downstream of follow_branch_a and branch_false. The join task will show up as skipped because its trigger_rule is set to all_success by default, and the skip caused by t…
2. is_ai=False: A given blog article might have various statuses - for instance, it might be visible to everyone (i.e. public), or only visible to the author (i.e. private). It may also be hidd…
3. is_ai=False: You can adjust the configuration of cmake variables optionally (without building first), by doing the following. For example, adjusting the pre-detected directories for CuDNN or…
4. is_ai=False: When we submit the form, the POST /articles request is mapped to the create action. The create action does attempt to save @article. Therefore, validations are checked. If any v…

Frozen program/prompt consistency: {'checked': True, 'predictors'

## Apply the pattern

Adapt the compile shape above to your own DSPy program, metric, and frozen
train/validation split. Evaluate the returned program on a test set that was not
used during compilation, and compare accuracy, compile cost, and inference latency
rather than treating a single score as the whole result.

The complete Chapter 6 rerun is summarized in `CHAPTER_RESULTS.md`. Raw provider
transcripts and temporary training outputs are intentionally excluded from the
student download.